<a href="https://colab.research.google.com/github/lmendezayl/uba-optimizacion-tp1/blob/main/tp1_optimizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import Pkg; Pkg.add("RDatasets")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed Mocking ─────────── v0.8.1
   Installed TZJData ─────────── v1.5.0+2025b
   Installed RData ───────────── v1.1.0
   Installed CategoricalArrays ─ v1.1.0
   Installed TimeZones ───────── v1.22.2
   Installed RDatasets ───────── v0.8.1
  Installing 1 artifacts
   Installed artifact tzjdata                  112.5 KiB
    Updating `~/.julia/environments/v1.12/Project.toml`
  [ce6b1742] + RDatasets v0.8.1
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [324d7699] + CategoricalArrays v1.1.0
  [78c3b35d] + Mocking v0.8.1
  [df47a6cb] + RData v1.1.0
  [ce6b1742] + RDatasets v0.8.1
  [dc5dba14] + TZJData v1.5.0+2025b
  [f269a46b] + TimeZones v1.22.2
Precompiling packages...
   5694.3 ms  ✓ TZJData
   4883.3 ms  ✓ Mocking
   8031.2 ms  ✓ CategoricalArrays
   3161.1 ms  ✓ CategoricalArrays → CategoricalArraysJSONExt
   3625.5 ms  ✓ CategoricalArrays → CategoricalArraysStatsBaseExt


In [3]:
using LinearAlgebra, RDatasets, Random, Statistics, Plots

In [4]:
iris = dataset("datasets", "iris")
X = Matrix(iris[:, 1:4])
y = iris.Species

150-element CategoricalArrays.CategoricalArray{String,1,UInt8}:
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 ⋮
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"

In [5]:
# utils_tools.jl
function split(X, y; dims=1, ratio_train=0.8)
    n = length(y)

    n_train = round(Int, ratio_train*n) #Redondeamos el corte
    i_rand = randperm(n)				# Permutamos los indices
    i_train = i_rand[1:n_train] 		# Usamos el 80% para train
    i_test = i_rand[n_train+1:end] 		# El resto lo usamos para test

    X_train = X[i_train,:]
    y_train = y[i_train]
    X_test  = X[i_test, :]
    y_test  = y[i_test]
    return X_train, y_train, X_test, y_test

end

function normalize(X_train, X_test; dims=1)
    col_mean = mean(X_train; dims)
    col_std = std(X_train; dims)

    return (X_train .- col_mean) ./ col_std, (X_test .- col_mean) ./ col_std
end

function onehot(y, classes)
	y_onehot = zeros(length(classes), length(y))
	num_of_class = 1:length(classes)

	for i in 1:length(y)
		y_onehot[:,i] = y[i].==classes
	end
	return y_onehot
end

function prepare_data(X, y; do_normal=true, do_onehot=true, kwargs...)
    X_train, y_train, X_test, y_test = split(X, y)

    if do_normal
        X_train, X_test = normalize(X_train, X_test; kwargs...)
    end

    classes = unique(y)

    if do_onehot
        y_train = onehot(y_train, classes)
        y_test = onehot(y_test, classes)
    end

    return X_train', y_train, X_test', y_test, classes
end

X_train, y_train, X_test, y_test, classes = prepare_data(X, y)

mean_tuple(d::AbstractArray{<:Tuple}) = Tuple([mean([d[k][i] for k in 1:length(d)]) for i in 1:length(d[1])])

grad_total(modelo,grad,x,y) = mean_tuple([grad(modelo, X_train[:,k], y_train[:,k]) for k in 1:size(X_train,2)])

# funciones de activacion
relu(x) = max.(0,x)
id(x) = x
softmax(x) = exp.(x) ./ sum(exp.(x), dims=1)

softmax (generic function with 1 method)

In [6]:
mutable struct RedNeuronal{T<:Real}
	# Utilizaremos una red neuronal con tres capas con funciones de activacion ReLU,
	# identidad y softmax de 5 niveles.
	#
	# Nota: no es como Python, no hace falta definir una clase, ni tapoco
	# hace falta definir atributos como __init__ para poder definir el llamado a
	# el constructor de esta clase. Aca directamente llamamos
	# m = RedNeuronal(W1, b1, W2, b2)
	# y listo, con esto es suficiente para crear el modelo.
	#
	# El downside es que no podemos crear metodos bajo el struct RedNeuronal
	W1::Matrix{T} # l1 -> l2
	b1::Vector{T}
	W2::Matrix{T} # l2 -> l3
	b2::Vector{T}
end

# creamos el modelo con el constructor de Red Neuronal
Random.seed!(73)
W1 = randn(5, size(X_train, 1)) # este 1 era un 2, tomando una matriz de 5x4 enlugar de 5x120!
b1 = randn(5)
W2 = randn(size(y_train, 1), 5)
b2 = randn(size(y_train, 1 ))

model = RedNeuronal(W1, b1, W2, b2)

RedNeuronal{Float64}([1.2123689466501288 -1.8659948470385643 1.5359646820970976 -2.946693261135319; 0.07867339151355215 -0.5409383051722126 1.904244155888974 0.4000651169550351; … ; -1.4081384368973173 0.3566138060639536 0.758845608555424 -0.2462781339520417; 0.4644730607385192 -0.4580210630616379 -2.3363570247525765 0.6032439841236702], [0.036509117320801934, 2.0183941863348958, -1.4222121770628655, -0.02458224672669713, 0.5662594627526059], [0.9865857065343743 0.5569751877519062 … -0.013964807677120253 -1.1131349472530667; 0.5499123327973198 0.12834599681305806 … -0.7764680654747667 0.36181588423766176; -0.7411473285232308 -0.06050648910140387 … -0.7658544536287244 2.1530628891736723], [-0.14226841165752968, -1.0368122047759714, -0.14131008223303249])

In [7]:
function forward_pass(model, X) # Done
	# Implementar una funcion que para cada dato x retorne la prediccion segun
	# el modelo.
	W1, b1, W2, b2 = model.W1, model.b1, model.W2, model.b2
	z_1 = W1 * X .+ b1
	size(z_1)
	a_1 = relu(z_1)
	z_2 = W2 * a_1 .+ b2
	a_2 = z_2
	y = softmax(a_2)
	return y
end

forward_pass (generic function with 1 method)

In [36]:
forward_pass(model, X_train)

3×120 Matrix{Float64}:
 0.000963536  0.879318   0.00161007  …  0.988723     0.924154   0.673682
 0.00631827   0.0576534  0.0081156      0.0110909    0.0481644  0.13899
 0.992718     0.0630289  0.990274       0.000186133  0.0276817  0.187328

In [23]:
# TODO
function grad(model, X, y)
	# Implementar una funcion que calcule el gradiente de la funcion de perdida para
	# cada par de datos (x,y) utilizando backprop. Usarla para calcular el grad completo
	# usando la funcion grad_total de utils_tools.jl

	W1, b1, W2, b2 = model.W1, model.b1, model.W2, model.b2

    z_1 = W1 * X .+ b1
    a_1 = relu.(z_1)
    z_2 = W2 * a_1 .+ b2
    y_pred = softmax(z_2)

	relu_prime(X) = X > 0 ? 1.0 : 0.0 # heaviside = relu'

    delta_2 = y_pred .- y # https://eli.thegreenplace.net/2016/the-softmax-function-and-its-derivative/
    grad_W2 = delta_2 * a_1'
    grad_b2 = delta_2
    delta_1 = (W2' * delta_2) .* relu_prime.(z_1) # .* es prod de hadamard
    grad_W1 = delta_1 * X'
    grad_b1 = delta_1
    return (grad_W1, grad_b1, grad_W2, grad_b2)
end

grad (generic function with 1 method)

In [24]:
grad(model, X_train, y_train)

([47.011639493752675 -25.204746186402375 48.69737732391013 40.65821970295488; 22.953730059529388 -13.856069849591199 27.881941174686663 27.28751225339785; … ; 23.895046522857694 -38.22192603849837 44.07595647297686 42.39534726353703; -134.49797495090334 160.48134848283738 -188.26000339194192 -182.3250192247263], [-1.7179110380457878 0.0 … 1.6588743763861438 0.052327884763827555; -0.0 0.5538505688388704 … 0.5797440756351585 0.2533825656129426; … ; -0.7512322329915663 0.0 … 0.0 0.5156731056029655; 3.251733154437263 -0.0 … -0.0 -0.6580980101754524], [74.56931131773959 269.20044763799785 … -31.911107813488115 -64.74278035940522; -53.322328353444355 -93.45593583604607 … -5.462986390877098 -13.937199563050502; -21.246982964295235 -175.74451180195166 … 37.3740942043652 78.67997992245571], [-0.9990364641466799 0.8793177156404166 … 0.9241539089913523 0.6736822133625038; 0.006318270589625813 0.057653417934543164 … 0.048164418893361836 -0.8610098849005081; 0.9927181935570543 -0.9369711335749599 …

In [51]:
# TODO
function train!(model, X_train, y_train, step=1e-2, max_iter=500)
	# La funcion debe retornar el modelo entrenado y un vector con los valores de la
	# funcion de perdida en cada iteracion
	loss_hist = []
	cross_entropy(y_pred) = -sum.(y_train * log.(y_pred))

	for i in 1:max_iter
		# begin iter
		# correr forward pass -> y
		# calcular cross entropy -> loss
		# hacer backprop -> delta_C
		# actualizar? preguntar
		# end iter
		y_pred = forward_pass(model, X_train)
		println(size(y_pred))
		loss = cross_entropy(y_pred)
		push!(loss_hist, loss)
		grad_W1, grad_b1, grad_W2, grad_b2 = grad_total(model, grad, X_train, y_train)
		model.W1 .-= step .* grad_W1
		model.b1 .-= step .* grad.b1
		model.W2 .-= step .* grad.W2
		model.b2 .-= step .* grad.b2

		if i % 50 == 0 || i == 1
            println("Iteración $i | Loss: $current_loss")
        end

	end

	return model, loss_hist
end

train! (generic function with 4 methods)

In [52]:
model, loss_hist = train!(model, X_train, y_train)

(3, 120)


LoadError: DimensionMismatch: incompatible dimensions for matrix multiplication: tried to multiply a matrix of size (3, 120) with a matrix of size (3, 120). The second dimension of the first matrix: 120, does not match the first dimension of the second matrix: 3.

In [116]:
# TODO
function results()
	# Implementar una funcion que retorne un grafico de los valores de la funcion
	# de perdida para cada iteracion y el rendimiento de nuestro modelo tanto en
	# los conjuntos de entrenamiento como en los de prueba, por ejemplo calculando
	#
	# 				 # favorables
	# 		       		---------------
	# 		      		# casos totales

end

results (generic function with 1 method)

In [117]:
# Cada metodo debe retornar:
# 	Estado (convegio, maximas iteraciones alcanzadas, no convergio)
# 	Numero de iteraciones realizadas
# 	Valor final de ||grad(f)||
# TODO
function gradiente_const()
end


gradiente_const (generic function with 1 method)

In [118]:
# TODO
function gradiente_armijo()
end

gradiente_armijo (generic function with 1 method)

In [119]:
# TODO
function newton_raphson()
end

newton_raphson (generic function with 1 method)

In [120]:
# TODO
function gradiente_conj_no_lineal()
end

gradiente_conj_no_lineal (generic function with 1 method)

In [121]:
# TODO
function bfgs()
end

bfgs (generic function with 1 method)

In [122]:
# TODO
function dolan_more()
	# construir perfil de desempeno de dolan & more para la metrica de iteraciones.
	# graficar y realizar un breve analisis.
	# sug: el primer paso consiste en ejecutar cada uno de los algoritmos sobre cada
	# problema y registrr el costo (iteraciones) de cada uno. Asi, podemos formar la
	# matriz C de costos. Luego se puede calcular una matriz R con la formula de r_sp
	# (ver clase de perfil de desempeno)
end

dolan_more (generic function with 1 method)